
# ML Experiment Intelligence Suite
## Project Brief

---

### Background

Meridian Analytics' data science team ships 15–20 model experiments a week across churn, fraud, pricing, and forecasting. Review before deployment is manual and inconsistent: a senior data scientist skims a notebook, checks a couple of metrics, and approves — or doesn't, depending on how close it is to a deadline. Two incidents in the last two quarters made it to production before anyone caught them: a churn model trained on a target-leaking feature, and a credit model with a demographic parity gap that surfaced in a compliance audit, not in review.

Leadership wants a review system that applies the same rigor to every experiment, every time, and makes a senior reviewer's judgment call the last step — not the only step.

### Objective

Build an ML experiment intelligence suite that, given a submitted experiment card:

1. Validates and parses the submission against a fixed schema.
2. Retrieves similar past experiments — successes and failures alike — for context.
3. Runs three independent diagnostic checks: overfitting/generalization, data leakage, and bias/statistical validity.
4. Produces a structured review report with an explicit verdict and reasoning a human can audit.
5. Routes deployment through a senior-reviewer approval gate, with a bounded revise-and-resubmit loop.
6. Exposes the review pipeline through an MCP server so CI/CD can submit experiments and poll for a verdict without a human in that loop.

### Scope and Constraints

- No diagnostic or review step may invent a metric value. If a field isn't in the experiment card, it is reported as missing — never estimated.
- An experiment with a high-confidence leakage finding or a demographic bias flag is never silently approved, even under an `approve_with_monitoring` verdict.
- Every verdict must cite the specific evidence (a metric, a feature name, a prior experiment) that produced it — "looks fine" is not a valid reasoning trace.

### Deliverables

```
capstone2/
├── data/
│   ├── experiments.json         # Experiment knowledge base (provided)
│   └── model_registry.json      # Production model registry (provided)
├── rag/
│   ├── ingest.py
│   └── retriever.py
├── agent/
│   ├── tools.py                 # metric_analyser, leakage_detector, bias_checker
│   ├── prompts.py
│   └── graph.py
├── mcp_server/
│   └── server.py
├── api/
│   └── main.py
├── Dockerfile
├── docker-compose.yml
└── README.md                    # Design memo (see Architecture section)
```

**Submission:** repository link, plus a 2–3 minute recording of an end-to-end review including one rejection-with-feedback cycle.



---
## Architecture

```
                       EXPERIMENT CARD
                             │
                             ▼
                   ┌─────────────────┐
                   │  INGEST         │  Validates the submitted experiment card
                   └───────┬─────────┘
                           ▼
                   ┌─────────────────┐
                   │  RETRIEVE       │  Hybrid BM25 + vector search over past experiments
                   └───────┬─────────┘
                           ▼
                   ┌─────────────────┐
                   │  ANALYSE        │  metric_analyser, leakage_detector, bias_checker
                   └───────┬─────────┘
                           ▼
                   ┌─────────────────┐
                   │  REVIEW         │  Structured verdict + reasoning (may be skipped — see below)
                   └───────┬─────────┘
                   ┌───────▼─────────┐
                   │  APPROVAL GATE  │  Senior reviewer approves / rejects with feedback
                   └───────┬─────────┘
               approved    │  rejected → back to REVIEW (bounded retries)
                           ▼
                   ┌─────────────────┐
                   │  DEPLOY         │  Registers to model registry with lineage
                   └─────────────────┘
                           │
                   ┌───────▼─────────┐
                   │  MCP SERVER     │  CI/CD submission + polling interface
                   └─────────────────┘
```

**Fast path:** if `analyse` returns a blocking leakage or bias finding, route straight to the approval gate with that finding attached, skipping `review`. A confirmed leakage risk doesn't need a generated narrative to justify rejecting it — it needs to reach a human immediately.

### Design memo

Your README must include a short design memo (300–500 words) answering:

1. Why run leakage detection as a discrete tool call rather than folding it into the review prompt?
2. Should a bias finding ever block deployment outright, or should it always route to a human first? Justify your choice.
3. What do you embed for retrieval — raw metrics, or derived descriptions (algorithm, task, outcome, failure pattern)? Why?



---
## Reference Data

Run the cell below once to materialize the experiment knowledge base and model registry. Schema is fixed — your retrieval and diagnostic logic must work against it as-is.


In [1]:
import json, os
os.makedirs("data", exist_ok=True)

EXPERIMENTS = [
    {"id":"EXP-001","model":"churn_lr_v1","algorithm":"LogisticRegression","task":"binary_classification",
     "train_acc":0.82,"val_acc":0.81,"test_acc":0.80,"roc_auc":0.88,"f1":0.79,
     "train_size":120000,"val_size":15000,"test_size":15000,
     "features":["recency","frequency","monetary","days_since_last_purchase"],
     "leakage_risk":"none","bias_flags":[],"outcome":"approved","deployment_date":"2024-01-15",
     "notes":"Baseline model. Good generalisation.",
     "tags":["churn","baseline","logistic"]},
    {"id":"EXP-002","model":"churn_xgb_v1","algorithm":"XGBoost","task":"binary_classification",
     "train_acc":0.99,"val_acc":0.71,"test_acc":0.69,"roc_auc":0.84,"f1":0.72,
     "train_size":120000,"val_size":15000,"test_size":15000,
     "features":["recency","frequency","monetary","future_purchase_flag"],
     "leakage_risk":"high","bias_flags":["future_purchase_flag is a target leaker"],
     "outcome":"rejected","deployment_date":None,
     "notes":"Target leakage detected: future_purchase_flag derives from label.",
     "tags":["churn","overfit","leakage","rejected"]},
    {"id":"EXP-003","model":"fraud_rf_v2","algorithm":"RandomForest","task":"binary_classification",
     "train_acc":0.96,"val_acc":0.95,"test_acc":0.94,"roc_auc":0.98,"f1":0.91,
     "train_size":500000,"val_size":60000,"test_size":60000,
     "features":["amount","merchant_category","hour","device_fingerprint","velocity_1h"],
     "leakage_risk":"none","bias_flags":["underrepresented_region: APAC only 3% of training"],
     "outcome":"approved_with_monitoring","deployment_date":"2024-03-01",
     "notes":"Approved with APAC bias flag — additional monitoring required.",
     "tags":["fraud","production","bias-flag"]},
    {"id":"EXP-004","model":"ltv_dnn_v1","algorithm":"DNN","task":"regression",
     "train_acc":0.91,"val_acc":0.90,"test_acc":0.89,"roc_auc":None,"f1":None,
     "train_size":80000,"val_size":10000,"test_size":10000,
     "features":["age","tenure","product_tier","avg_order_value","support_tickets"],
     "leakage_risk":"none","bias_flags":[],"outcome":"approved","deployment_date":"2024-04-10",
     "notes":"Strong generalisation. No issues.","tags":["ltv","regression","production"]},
    {"id":"EXP-005","model":"recommender_mf_v1","algorithm":"MatrixFactorization","task":"ranking",
     "train_acc":0.78,"val_acc":0.61,"test_acc":0.59,"roc_auc":None,"f1":None,
     "train_size":1000000,"val_size":100000,"test_size":100000,
     "features":["user_id","item_id","interaction_type","timestamp"],
     "leakage_risk":"medium","bias_flags":["cold_start: 8200 new users have 0 interactions"],
     "outcome":"rejected","deployment_date":None,
     "notes":"Significant val/test gap (val=0.61). Cold start not handled. Rejected.",
     "tags":["recommender","overfit","cold-start","rejected"]},
    {"id":"EXP-006","model":"nps_sentiment_v1","algorithm":"BERT-finetune","task":"sentiment_classification",
     "train_acc":0.94,"val_acc":0.93,"test_acc":0.92,"roc_auc":0.97,"f1":0.91,
     "train_size":50000,"val_size":6000,"test_size":6000,
     "features":["survey_text","product_category","channel"],
     "leakage_risk":"none","bias_flags":[],"outcome":"approved","deployment_date":"2024-06-01",
     "notes":"Good generalisation across product categories.","tags":["nlp","sentiment","bert"]},
    {"id":"EXP-007","model":"demand_forecast_v3","algorithm":"LightGBM","task":"regression",
     "train_acc":0.88,"val_acc":0.87,"test_acc":0.51,"roc_auc":None,"f1":None,
     "train_size":365000,"val_size":30000,"test_size":30000,
     "features":["day_of_week","month","holiday_flag","promo_flag","lag_7","lag_28"],
     "leakage_risk":"high","bias_flags":[],
     "outcome":"rejected","deployment_date":None,
     "notes":"Val/test catastrophic gap (0.87 → 0.51). Temporal leakage: lag features computed on full dataset before split.",
     "tags":["forecast","temporal-leakage","rejected"]},
    {"id":"EXP-008","model":"credit_score_v2","algorithm":"GradientBoosting","task":"binary_classification",
     "train_acc":0.87,"val_acc":0.86,"test_acc":0.85,"roc_auc":0.91,"f1":0.84,
     "train_size":200000,"val_size":25000,"test_size":25000,
     "features":["income","employment_length","debt_to_income","payment_history","credit_utilisation"],
     "leakage_risk":"none","bias_flags":["demographic_parity: approval rate Black applicants 18pp lower than White"],
     "outcome":"rejected_bias","deployment_date":None,
     "notes":"Rejected due to demographic parity violation — regulatory risk.",
     "tags":["credit","fairness","bias","rejected","regulatory"]},
    {"id":"EXP-009","model":"inventory_optim_v1","algorithm":"XGBoost","task":"regression",
     "train_acc":0.85,"val_acc":0.84,"test_acc":0.83,"roc_auc":None,"f1":None,
     "train_size":180000,"val_size":20000,"test_size":20000,
     "features":["sku","warehouse","lead_time","safety_stock","demand_30d"],
     "leakage_risk":"none","bias_flags":[],"outcome":"approved","deployment_date":"2024-09-15",
     "notes":"Strong results. Deployed to ops team.","tags":["inventory","xgboost","production"]},
    {"id":"EXP-010","model":"price_elasticity_v1","algorithm":"LinearRegression","task":"regression",
     "train_acc":0.61,"val_acc":0.59,"test_acc":0.58,"roc_auc":None,"f1":None,
     "train_size":45000,"val_size":5000,"test_size":5000,
     "features":["price","competitor_price","category","season"],
     "leakage_risk":"none","bias_flags":[],"outcome":"rejected_performance","deployment_date":None,
     "notes":"Below performance threshold (target: val >= 0.70). Needs more features.",
     "tags":["pricing","underperforming","rejected"]},
    {"id":"EXP-011","model":"churn_xgb_v2","algorithm":"XGBoost","task":"binary_classification",
     "train_acc":0.89,"val_acc":0.88,"test_acc":0.87,"roc_auc":0.93,"f1":0.86,
     "train_size":150000,"val_size":18000,"test_size":18000,
     "features":["recency","frequency","monetary","days_since_last_purchase","support_tickets"],
     "leakage_risk":"none","bias_flags":[],"outcome":"approved","deployment_date":"2024-11-01",
     "notes":"Improved churn model. No leakage. Approved.","tags":["churn","xgboost","production"]},
    {"id":"EXP-012","model":"anomaly_iforest_v1","algorithm":"IsolationForest","task":"anomaly_detection",
     "train_acc":None,"val_acc":0.91,"test_acc":0.89,"roc_auc":0.94,"f1":0.88,
     "train_size":300000,"val_size":35000,"test_size":35000,
     "features":["transaction_amount","frequency","merchant_type","geo_distance"],
     "leakage_risk":"none","bias_flags":[],"outcome":"approved","deployment_date":"2024-12-15",
     "notes":"Unsupervised — no train_acc. Strong recall on fraud. Approved.","tags":["anomaly","unsupervised","production"]},
    {"id":"EXP-013","model":"text_classifier_v1","algorithm":"DistilBERT","task":"multi_class_classification",
     "train_acc":0.95,"val_acc":0.94,"test_acc":0.93,"roc_auc":0.98,"f1":0.93,
     "train_size":80000,"val_size":10000,"test_size":10000,
     "features":["ticket_text","category_label"],
     "leakage_risk":"low","bias_flags":["category_label was preprocessed on full dataset — minor leakage risk"],
     "outcome":"approved_with_monitoring","deployment_date":"2025-01-10",
     "notes":"Minor preprocessing leakage. Approved with note to fix in v2.","tags":["nlp","classification","monitoring"]},
    {"id":"EXP-014","model":"survival_model_v1","algorithm":"CoxPH","task":"survival_analysis",
     "train_acc":None,"val_acc":0.77,"test_acc":0.76,"roc_auc":0.83,"f1":None,
     "train_size":60000,"val_size":8000,"test_size":8000,
     "features":["age","product_tier","tenure","payment_method"],
     "leakage_risk":"none","bias_flags":[],"outcome":"approved","deployment_date":"2025-02-01",
     "notes":"Survival model — C-index 0.77. Approved for churn timing prediction.","tags":["survival","churn","timing"]},
    {"id":"EXP-015","model":"ab_test_uplift_v1","algorithm":"UpliftForest","task":"uplift_modelling",
     "train_acc":None,"val_acc":0.62,"test_acc":0.61,"roc_auc":None,"f1":None,
     "train_size":40000,"val_size":5000,"test_size":5000,
     "features":["treatment_flag","user_segment","channel","propensity_score"],
     "leakage_risk":"high","bias_flags":["propensity_score computed post-treatment — selection bias"],
     "outcome":"rejected","deployment_date":None,
     "notes":"propensity_score is post-treatment — cannot be used as feature.","tags":["uplift","causal","leakage","rejected"]},
    {"id":"EXP-016","model":"forecast_seasonal_v1","algorithm":"Prophet","task":"time_series",
     "train_acc":None,"val_acc":0.84,"test_acc":0.83,"roc_auc":None,"f1":None,
     "train_size":730,"val_size":90,"test_size":90,
     "features":["date","sales","holiday","promo"],
     "leakage_risk":"none","bias_flags":[],"outcome":"approved","deployment_date":"2025-03-01",
     "notes":"Seasonal decomposition works well. MAPE=8.2%. Approved.","tags":["forecast","prophet","seasonal","production"]},
    {"id":"EXP-017","model":"embeddings_v1","algorithm":"sentence-transformers","task":"embedding",
     "train_acc":None,"val_acc":0.89,"test_acc":0.88,"roc_auc":None,"f1":None,
     "train_size":500000,"val_size":50000,"test_size":50000,
     "features":["document_text","document_type"],
     "leakage_risk":"none","bias_flags":["domain_coverage: legal docs only 2% of corpus"],
     "outcome":"approved_with_monitoring","deployment_date":"2025-03-15",
     "notes":"Deployed for semantic search. Legal domain underrepresented.","tags":["embeddings","rag","domain-bias"]},
    {"id":"EXP-018","model":"causal_impact_v1","algorithm":"CausalImpact","task":"causal_inference",
     "train_acc":None,"val_acc":0.71,"test_acc":0.69,"roc_auc":None,"f1":None,
     "train_size":365,"val_size":30,"test_size":30,
     "features":["daily_revenue","intervention_flag","control_market"],
     "leakage_risk":"none","bias_flags":["small_test_size: 30 days insufficient for statistical power"],
     "outcome":"rejected_stats","deployment_date":None,
     "notes":"Test set too small for reliable causal estimates. Extend observation window.",
     "tags":["causal","underpowered","rejected"]},
    {"id":"EXP-019","model":"rl_pricing_v1","algorithm":"PPO","task":"reinforcement_learning",
     "train_acc":None,"val_acc":0.73,"test_acc":0.71,"roc_auc":None,"f1":None,
     "train_size":1000000,"val_size":None,"test_size":100000,
     "features":["state_price","competitor_price","inventory_level","demand_signal"],
     "leakage_risk":"none","bias_flags":["simulation_gap: training env != production distribution"],
     "outcome":"approved_limited","deployment_date":"2025-04-01",
     "notes":"Deployed to 5% of traffic only. Simulation gap requires live monitoring.","tags":["rl","pricing","simulation","limited"]},
    {"id":"EXP-020","model":"multimodal_v1","algorithm":"CLIP-finetune","task":"image_text_matching",
     "train_acc":0.91,"val_acc":0.90,"test_acc":0.89,"roc_auc":0.96,"f1":0.89,
     "train_size":200000,"val_size":25000,"test_size":25000,
     "features":["image_embedding","product_title","category"],
     "leakage_risk":"none","bias_flags":[],"outcome":"approved","deployment_date":"2025-05-01",
     "notes":"Strong multimodal results. Deployed for visual search.","tags":["multimodal","clip","production"]}
]

MODEL_REGISTRY = [
    {"id":"MDL-001","name":"churn_lr_v1","status":"production","version":"1.0","experiment_id":"EXP-001","deployed_by":"ds-team@co.com","sla_latency_ms":50},
    {"id":"MDL-002","name":"churn_xgb_v2","status":"production","version":"2.0","experiment_id":"EXP-011","deployed_by":"ds-team@co.com","sla_latency_ms":30},
    {"id":"MDL-003","name":"fraud_rf_v2","status":"production","version":"2.0","experiment_id":"EXP-003","deployed_by":"risk-eng@co.com","sla_latency_ms":20},
    {"id":"MDL-004","name":"ltv_dnn_v1","status":"production","version":"1.0","experiment_id":"EXP-004","deployed_by":"ds-team@co.com","sla_latency_ms":100},
    {"id":"MDL-005","name":"nps_sentiment_v1","status":"production","version":"1.0","experiment_id":"EXP-006","deployed_by":"ds-team@co.com","sla_latency_ms":200},
    {"id":"MDL-006","name":"inventory_optim_v1","status":"production","version":"1.0","experiment_id":"EXP-009","deployed_by":"ops-eng@co.com","sla_latency_ms":80},
    {"id":"MDL-007","name":"anomaly_iforest_v1","status":"production","version":"1.0","experiment_id":"EXP-012","deployed_by":"risk-eng@co.com","sla_latency_ms":15},
    {"id":"MDL-008","name":"forecast_seasonal_v1","status":"production","version":"1.0","experiment_id":"EXP-016","deployed_by":"ds-team@co.com","sla_latency_ms":500},
    {"id":"MDL-009","name":"embeddings_v1","status":"production","version":"1.0","experiment_id":"EXP-017","deployed_by":"platform@co.com","sla_latency_ms":50},
    {"id":"MDL-010","name":"rl_pricing_v1","status":"limited","version":"1.0","experiment_id":"EXP-019","deployed_by":"pricing@co.com","sla_latency_ms":10}
]


In [2]:
with open("data/experiments.json","w") as f: json.dump(EXPERIMENTS, f, indent=2)
with open("data/model_registry.json","w") as f: json.dump(MODEL_REGISTRY, f, indent=2)
print(f"✓ {len(EXPERIMENTS)} experiments written")
print(f"✓ {len(MODEL_REGISTRY)} models in registry")
outcomes = {}
for e in EXPERIMENTS:
    outcomes[e["outcome"]] = outcomes.get(e["outcome"],0)+1
print("Outcome distribution:", outcomes)

✓ 20 experiments written
✓ 10 models in registry
Outcome distribution: {'approved': 9, 'rejected': 4, 'approved_with_monitoring': 3, 'rejected_bias': 1, 'rejected_performance': 1, 'rejected_stats': 1, 'approved_limited': 1}



---
## Retrieval Layer

**What to embed:** a description combining algorithm, task, outcome, tags, and notes — not raw metrics. Metrics belong in metadata for filtering; semantic similarity should capture *experiment type and failure pattern*, not a numeric closeness that happens to be superficial.

**Specification**
- BM25 target: exact algorithm/task matches (`XGBoost`, `target leakage`, `temporal`).
- Vector target: conceptual similarity (`demand forecasting` ↔ `time series lag features`).
- After RRF fusion, exclude any experiment sharing the query's own `model` name — you're looking for precedent, not your own submission reflected back.

**Implementation note:** ChromaDB's `where` filter (`collection.query(where={'outcome': {'$ne': 'approved'}})`) is useful for surfacing failure patterns specifically — worth exposing as an option in `retriever.py` rather than always searching the full corpus.


In [ ]:
from rank_bm25 import BM25Okapi
import chromadb
from chromadb.utils import embedding_functions
import json, re

experiments = json.load(open("data/experiments.json"))

def tokenise(text): return re.findall(r'\w+', text.lower())

# Embed: algorithm + task + outcome + tags + notes (semantic signal)
def make_doc(e):
    return (f"{e['algorithm']} {e['task']} outcome={e['outcome']} "
            f"leakage={e['leakage_risk']} tags={' '.join(e['tags'])} notes={e['notes']}")

bm25_corpus = [tokenise(make_doc(e)) for e in experiments]
bm25 = BM25Okapi(bm25_corpus)

chroma_client = chromadb.Client()
ef = embedding_functions.DefaultEmbeddingFunction()
collection = chroma_client.get_or_create_collection("experiments", embedding_function=ef)

collection.add(
    documents=[make_doc(e) for e in experiments],
    metadatas=[{"id":e["id"],"outcome":e["outcome"],"leakage_risk":e["leakage_risk"],
                "algorithm":e["algorithm"],"task":e["task"]} for e in experiments],
    ids=[e["id"] for e in experiments]
)
print(f"✓ BM25 + vector store built — {len(experiments)} experiments")

def hybrid_retrieve_experiments(query: str, top_k: int = 3,
                                 exclude_id: str = None, k: int = 60) -> list:
    """RRF fusion of BM25 + vector, with optional self-exclusion filter."""
    # TODO: score `query` against `bm25`, rank descending, excluding `exclude_id`
    # TODO: query `collection` for vector-ranked ids (n_results=len(experiments)), excluding `exclude_id`
    # TODO: fuse both rankings with RRF and return the top_k experiments with their rrf_score
    raise NotImplementedError("hybrid_retrieve_experiments not implemented")

# Smoke test
q = "XGBoost binary classification with potential target leakage in features"
results = hybrid_retrieve_experiments(q)
print(f"\nQuery: {q}")
for r in results:
    e = r["experiment"]
    print(f"  [{r['rrf_score']}] {e['id']} — {e['model']} ({e['outcome']}) leakage={e['leakage_risk']}")

# 🎯 HINT: Try querying for bias-related experiments:
# hybrid_retrieve_experiments("credit scoring model with demographic disparity fairness concern")
# Do the bias-rejected experiments surface at the top?


---
## Diagnostic Tools

Three tools, each responsible for one class of risk — keep them independent so a reviewer can trust each finding on its own:

| Tool | Detects | Threshold |
|---|---|---|
| `metric_analyser` | Overfitting, underperformance, distribution shift | Train–val gap > 0.15 is high risk; val–test delta > 0.05 is a shift warning |
| `leakage_detector` | Target, temporal, and preprocessing leakage | Rule-based `leakage_risk` field plus LLM reasoning over feature names against task type |
| `bias_checker` | Demographic bias, domain coverage gaps, insufficient statistical power | Explicit `bias_flags`, plus test-set size below ~1,000 per class flagged as underpowered |

`leakage_detector` and `bias_checker` both set `blocks_deployment` — this flag drives the fast-path routing in the graph, so it needs to be conservative and well-justified, not a soft signal.


In [ ]:
import os, json
from openai import OpenAI
from typing import Optional

endpoint        = os.getenv("endpoint")
deployment_name = os.getenv("deployment_name")
api_key         = os.getenv("OPENAI_API_KEY")
client = OpenAI(base_url=endpoint, api_key=api_key)

def get_llm_response(prompt: str) -> str:
    response = client.responses.create(model=deployment_name, input=prompt)
    return response.output[0].content[0].text

# ── TOOL 1: metric_analyser ──────────────────────────────────────────────────
def metric_analyser(experiment: dict) -> dict:
    """
    Analyses model metrics for overfitting, underfitting, and distribution shift.
    Returns structured risk assessment with numeric evidence.
    """
    # TODO: compute train-val gap; flag "high" risk if > 0.15, "medium" if > 0.08
    # TODO: compute val-test delta; flag distribution shift if abs(delta) > 0.05
    # TODO: flag underperformance if val_acc < 0.65, low discriminative power if roc_auc < 0.70
    # TODO: return {"tool", "risk_level", "overfit_gap", "val_test_delta", "findings"}
    raise NotImplementedError("metric_analyser not implemented")

# ── TOOL 2: leakage_detector ─────────────────────────────────────────────────
def leakage_detector(experiment: dict) -> dict:
    """
    Uses rule-based checks + LLM reasoning to detect data leakage patterns.
    Covers target leakage, temporal leakage, and preprocessing leakage.
    """
    # TODO: read the experiment's explicit leakage_risk field as one signal
    # TODO: ask the LLM to reason over feature names vs. task type for target/temporal/
    #       preprocessing leakage; parse its JSON response defensively
    # TODO: combine both signals into a final risk_level and set blocks_deployment when risk == "high"
    raise NotImplementedError("leakage_detector not implemented")

# ── TOOL 3: bias_checker ──────────────────────────────────────────────────────
def bias_checker(experiment: dict) -> dict:
    """
    Checks for demographic bias, domain coverage gaps, and statistical power issues.
    Returns risk level and actionable flags.
    """
    # TODO: flag low statistical power when test_size < 1000 (and critical when < 100)
    # TODO: classify each entry in bias_flags as a fairness/regulatory risk, a coverage gap, or generic
    # TODO: return {"tool", "risk_level", "blocks_deployment", "findings", "requires_monitoring"}
    raise NotImplementedError("bias_checker not implemented")

# ── Smoke test all 3 tools ────────────────────────────────────────────────────
test_exp = experiments[1]  # EXP-002: known leakage case
print(f"Testing on: {test_exp['id']} — {test_exp['model']}")
print("\n── metric_analyser ──")
print(json.dumps(metric_analyser(test_exp), indent=2))
print("\n── leakage_detector ──")
print(json.dumps(leakage_detector(test_exp), indent=2))
print("\n── bias_checker ──")
print(json.dumps(bias_checker(experiments[7]), indent=2))  # EXP-008: credit bias


---
## Prompting Strategy

| Prompt | Technique | Output |
|---|---|---|
| Experiment ingestor | Structured extraction with constraints | Validated `ExperimentCard` |
| Risk synthesizer | Role + chain-of-thought | `ReviewReport` — verdict, reasoning, itemized risks |
| Remediation advisor | Few-shot, grounded in the specific rejection | Concrete steps to fix and resubmit |

**Non-negotiable constraint:** never fabricate a metric value. If a field is absent from the experiment card, the report says so explicitly — the human reviewer may have the original run open in MLflow, and an invented number is worse than a missing one.


In [ ]:
from jinja2 import Template
from pydantic import BaseModel
from typing import List, Literal, Optional

class ExperimentCard(BaseModel):
    model_name:   str
    algorithm:    str
    task:         str
    train_acc:    Optional[float]
    val_acc:      Optional[float]
    test_acc:     Optional[float]
    roc_auc:      Optional[float]
    features:     List[str]
    train_size:   int
    test_size:    int

class ExperimentRisk(BaseModel):
    risk_type:    Literal["overfit","leakage","bias","underperformance","statistical_power","other"]
    severity:     Literal["low","medium","high","critical"]
    evidence:     str          # specific numeric evidence
    blocks_deployment: bool
    recommendation:    str

class ReviewReport(BaseModel):
    experiment_id:      str
    model_name:         str
    overall_verdict:    Literal["approve","approve_with_monitoring","reject","escalate"]
    verdict_reasoning:  str    # CoT trace for human reviewer
    risks:              List[ExperimentRisk]
    similar_experiments: List[str]  # EXP-XXX IDs
    deployment_conditions: List[str]
    estimated_fix_effort: Literal["none","hours","days","weeks"]

# ── PROMPT 1: Risk Synthesiser (Role + CoT) ───────────────────────────────────
# TODO: implement per the spec above — reason through metric quality, leakage, bias/fairness,
# statistical validity, and similar-experiment precedent before concluding a verdict. Reject on
# high leakage; reject or escalate on demographic bias.
REVIEW_TMPL = Template("""
{# TODO: role framing + 5-point reasoning framework + hard verdict rules + schema injection
   + human_feedback revision branch #}
""")

# ── PROMPT 2: Remediation Advisor (Few-shot) ─────────────────────────────────
# TODO: implement per the spec above — few-shot fix examples, numbered and actionable today,
# referencing the experiment's actual features/algorithm/metrics.
REMEDIATION_TMPL = Template("""
{# TODO: few-shot examples + rejected experiment / review findings / human feedback inputs #}
""")

print("✓ Prompt templates defined")
print(f"  ReviewReport fields: {list(ReviewReport.model_fields.keys())}")


---
## Orchestration

```
ingest → retrieve → analyse → review → [approval gate] → deploy
                                             │
                                 rejected ───┘  (re-review with feedback, bounded retries)
```

**State** is shared across every node — see `ExperimentState` in the cell below for the exact schema.

**Fast-path routing:** after `analyse`, check whether either diagnostic tool set `blocks_deployment`. If so, skip straight to the approval gate with the raw findings attached — generating a full narrative review for an experiment that's already disqualified burns latency and tokens without adding information the reviewer needs.

```python
def route_after_analyse(state):
    if state['leakage_result']['blocks_deployment'] or state['bias_result']['blocks_deployment']:
        return 'hitl_gate'
    return 'review'
```

**Routing after the approval gate:** `approved` → deploy; `rejected` with retries remaining → back to review with feedback attached; retries exhausted → stop and escalate.


In [ ]:
from langgraph.graph import StateGraph, END
from langgraph.checkpoint.memory import MemorySaver
from typing import TypedDict
from datetime import datetime
import json

class ExperimentState(TypedDict):
    raw_experiment:      dict
    parsed_card:         dict
    similar_experiments: list
    metric_result:       dict
    leakage_result:      dict
    bias_result:         dict
    review_report:       dict
    hitl_decision:       str
    human_feedback:      str
    approval_attempts:   int
    deployment_record:   dict
    audit_trail:         list

# ── NODE 1: Ingest ────────────────────────────────────────────────────────────
def node_ingest(state: ExperimentState) -> dict:
    """Parse and validate incoming experiment card."""
    # TODO: validate raw_experiment has the required fields (model_name, algorithm, task,
    #       val_acc, features, test_size), raising ValueError listing anything missing
    # TODO: append an audit_trail entry; return {"parsed_card", "audit_trail"}
    raise NotImplementedError("node_ingest not implemented")

# ── NODE 2: Retrieve ──────────────────────────────────────────────────────────
def node_retrieve(state: ExperimentState) -> dict:
    """Find similar past experiments for context."""
    # TODO: build a query from parsed_card, call hybrid_retrieve_experiments(), record in audit_trail
    raise NotImplementedError("node_retrieve not implemented")

# ── NODE 3: Analyse ───────────────────────────────────────────────────────────
def node_analyse(state: ExperimentState) -> dict:
    """Run all 3 diagnostic tools in sequence."""
    # TODO: call metric_analyser, leakage_detector, bias_checker on parsed_card
    # TODO: append an audit_trail entry summarising each tool's risk_level
    raise NotImplementedError("node_analyse not implemented")

# ── ROUTING: fast-path rejection ──────────────────────────────────────────────
def route_after_analyse(state: ExperimentState) -> str:
    """Skip the review node and go straight to HITL for critical leakage/bias."""
    # TODO: return "hitl_gate" if leakage_result or bias_result set blocks_deployment, else "review"
    raise NotImplementedError("route_after_analyse not implemented")

# ── NODE 4: Review ────────────────────────────────────────────────────────────
def node_review(state: ExperimentState) -> dict:
    """Generate structured review report using LLM."""
    # TODO: render REVIEW_TMPL (include human_feedback when present) and call get_llm_response()
    # TODO: parse the JSON response into ReviewReport, falling back to overall_verdict="escalate"
    raise NotImplementedError("node_review not implemented")

# ── NODE 5: HITL Gate ─────────────────────────────────────────────────────────
def node_hitl(state: ExperimentState) -> dict:
    """Display review report to human approver. Waits for update_state() injection."""
    # TODO: print the review (or the auto-reject reason, on the fast path) for the human reviewer
    # TODO: append an audit_trail entry; this node does not set hitl_decision itself
    raise NotImplementedError("node_hitl not implemented")

# ── NODE 6: Deploy ────────────────────────────────────────────────────────────
def node_deploy(state: ExperimentState) -> dict:
    """Register approved model to model registry with lineage metadata."""
    # TODO: build a deployment_record (status, verdict, conditions, similar_exps, timestamps)
    # TODO: append an audit_trail entry; return {"deployment_record", "audit_trail"}
    raise NotImplementedError("node_deploy not implemented")

# ── Routing ───────────────────────────────────────────────────────────────────
def route_after_hitl(state: ExperimentState) -> str:
    """approved -> deploy; rejected with retries left -> review; otherwise -> END (escalate)."""
    # TODO: implement the three-way routing described in the Orchestration spec above
    raise NotImplementedError("route_after_hitl not implemented")

# ── Build graph ───────────────────────────────────────────────────────────────
builder = StateGraph(ExperimentState)
builder.add_node("ingest",   node_ingest)
builder.add_node("retrieve", node_retrieve)
builder.add_node("analyse",  node_analyse)
builder.add_node("review",   node_review)
builder.add_node("hitl_gate",node_hitl)
builder.add_node("deploy",   node_deploy)

builder.set_entry_point("ingest")
builder.add_edge("ingest",   "retrieve")
builder.add_edge("retrieve", "analyse")
builder.add_conditional_edges("analyse", route_after_analyse,
    {"review":"review","hitl_gate":"hitl_gate"})
builder.add_edge("review",   "hitl_gate")
builder.add_conditional_edges("hitl_gate", route_after_hitl,
    {"deploy":"deploy","review":"review",END:END})
builder.add_edge("deploy",   END)

memory = MemorySaver()
graph  = builder.compile(checkpointer=memory, interrupt_before=["hitl_gate"])
print("✓ Experiment review graph compiled")


---
## End-to-End Runs

Exercise three paths before you consider this done: a clean approval, a leakage-triggered fast rejection, and a rejection-with-feedback that changes the resubmitted review.



---
## MCP Interface

This system has two very different callers — a CI/CD pipeline submitting experiments automatically, and a human reviewer making judgment calls — and the interface should reflect that.

| Type | Name | Caller |
|---|---|---|
| Tool | `submit_experiment` | Training job, at the end of a run |
| Tool | `submit_review_decision` | Senior reviewer, from Slack or a review UI |
| Tool | `get_review_status` | CI/CD, polling before it decides whether to proceed |
| Resource | `experiment_registry` | Read-only view of all reviewed experiments and outcomes |

**Auth:** two key tiers, not one — `X-Pipeline-Key` for CI/CD (write access to `submit_experiment` only) and `X-Reviewer-Key` for humans (access to decisions and status). A training job should never be able to approve its own experiment, even by accident.


In [ ]:
MCP_CODE = """
from mcp.server.fastmcp import FastMCP
from pydantic import BaseModel
from typing import Literal, Optional
import json, os, uuid

mcp = FastMCP("ml-experiment-review")

class ExperimentSubmission(BaseModel):
    model_name:   str
    algorithm:    str
    task:         str
    val_acc:      Optional[float]
    test_acc:     Optional[float]
    features:     list
    train_size:   int
    test_size:    int
    leakage_risk: str = "none"
    bias_flags:   list = []

class ReviewDecision(BaseModel):
    review_id:  str
    decision:   Literal["approved","rejected"]
    feedback:   str = ""

@mcp.tool()
def submit_experiment(experiment: ExperimentSubmission) -> dict:
    """Submit an ML experiment for automated review.
    Called by training CI/CD at end of run. Returns review_id for polling
    """
    review_id = f"rev-{uuid.uuid4().hex[:8]}"
    # In production: trigger graph.stream(initial_state, config)
    return {"review_id": review_id, "status": "review_started",
            "model_name": experiment.model_name,
            "message": f"Review started. Poll /status/{review_id} for updates."}

@mcp.tool()
def submit_review_decision(decision: ReviewDecision) -> dict:
    """Submit a human approval or rejection. Called by reviewer from Slack or UI."""
    config = {"configurable": {"thread_id": decision.review_id}}
    # graph.update_state(config, {"hitl_decision": decision.decision, ...})
    # graph.stream(None, config)
    return {"review_id": decision.review_id, "recorded": decision.decision,
            "status": "deploying" if decision.decision=="approved" else "rejected"}

@mcp.tool()
def get_review_status(review_id: str) -> dict:
    """Poll review status. CI/CD uses this to decide whether to deploy.
    Returns: pending | approved | approved_with_monitoring | rejected | escalated
    """
    config = {"configurable": {"thread_id": review_id}}
    # state = graph.get_state(config).values
    return {"review_id": review_id, "status": "pending_approval",
            "verdict": None, "can_deploy": False}

@mcp.resource("experiments://registry")
def experiment_registry() -> str:
    """Read-only registry of all reviewed experiments and their outcomes."""
    try:
        data = json.load(open("data/experiments.json"))
        return json.dumps({"experiments": data, "total": len(data)})
    except:
        return json.dumps({"experiments": [], "total": 0})

if __name__ == "__main__":
    mcp.run(transport="streamable-http", host="0.0.0.0", port=8001)
"""

import os
os.makedirs("mcp_server", exist_ok=True)
with open("mcp_server/server.py","w") as f:
    f.write(MCP_CODE.strip())
print("✓ mcp_server/server.py written")



---
## Deployment

**Required endpoints** (`api/main.py`): `POST /experiment`, `POST /review/{review_id}/decision`, `GET /review/{review_id}`, `GET /health`.

**Additional requirement for this project:** `GET /experiments/leaderboard`, returning approved experiments sorted by `val_acc` descending — this is the artifact the team actually checks day to day, so treat it as a first-class endpoint, not an afterthought.

Package as a multi-stage Docker build running as a non-root user, health check wired to `/health`.


In [ ]:
FASTAPI_CODE = """from fastapi import FastAPI, Header, HTTPException
from pydantic import BaseModel
from typing import Optional, List
import json, time, os

app = FastAPI(title="ML Experiment Review API", version="1.0")
START_TIME = time.time()
REVIEWS_PROCESSED = 0

class ExperimentRequest(BaseModel):
    model_name: str
    algorithm:  str
    task:       str
    val_acc:    Optional[float]
    test_acc:   Optional[float]
    features:   List[str]
    train_size: int
    test_size:  int

class DecisionRequest(BaseModel):
    decision: str   # "approved" | "rejected"
    feedback: str = ""

@app.post("/experiment")
async def submit_experiment(req: ExperimentRequest,
                             x_pipeline_key: Optional[str] = Header(None)):
    # TODO: trigger graph.stream(initial_state, config)
    raise HTTPException(status_code=501, detail="submit_experiment not implemented")

@app.post("/review/{review_id}/decision")
async def submit_decision(review_id: str, req: DecisionRequest,
                           x_reviewer_key: Optional[str] = Header(None)):
    # TODO: graph.update_state(...) then graph.stream(None, config)
    raise HTTPException(status_code=501, detail="submit_decision not implemented")

@app.get("/review/{review_id}")
async def get_review(review_id: str):
    # TODO: graph.get_state(config).values
    raise HTTPException(status_code=501, detail="get_review not implemented")

@app.get("/experiments/leaderboard")
async def leaderboard():
    # TODO: query approved/approved_with_monitoring experiments, sort by val_acc descending
    raise HTTPException(status_code=501, detail="leaderboard not implemented")

@app.get("/health")
async def health():
    return {"status":"ok","uptime_s":round(time.time()-START_TIME,1),
            "reviews_processed":REVIEWS_PROCESSED,
            "model":os.getenv("deployment_name","unknown")}
"""

import os
os.makedirs("api",exist_ok=True)
with open("api/main.py","w") as f: f.write(FASTAPI_CODE.strip())

DOCKERFILE = """# TODO: multi-stage Dockerfile per the Deployment spec above.
# Stage 1 (builder): FROM python:3.11-slim, install requirements.txt
# Stage 2 (runtime): FROM python:3.11-slim, create a non-root user (e.g. useradd -m -u 1000 appuser),
#                     copy the installed deps + app code from the builder stage, USER appuser,
#                     EXPOSE 8000, HEALTHCHECK hitting /health, CMD runs uvicorn as that user
"""
with open("Dockerfile","w") as f: f.write(DOCKERFILE.strip())
print("✓ api/main.py, Dockerfile written")